[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ai-forever/FRIDA-Decisions/blob/grok/examples-entity_alignment/examples/entity_alignment/entity_alignment.ipynb)

# Align two product cards

Two catalogues describe the same kettle in different words, or two different kettles that share a brand. "Are these the same product?" is a score. Two yes/no questions ask whether the brand matches and whether the model number matches. Code trusts the score: at or above 2.5 the pair is the same item, below 1.5 it is not, and the band in between waits for a person. The yes/no answers are printed next to the score and are not a veto, because a paraphrase can move them on its own.

The cards are plain sentences. The same facts as a JSON object stayed near the middle of the scale even when the model number matched.

Pattern: [Knowledge graph entity alignment](https://docs.typesafe.ai/cookbooks/entity_alignment). Cards are synthetic.


In [1]:
import sys
!{sys.executable} -m pip install -q "frida-decisions[torch] @ git+https://github.com/ai-forever/FRIDA-Decisions@v0.4.0"


In [2]:
import json, os
from pathlib import Path
import torch
import frida_decisions
from frida_decisions import Judge

MODEL = os.environ.get("FD_MODEL", "ai-forever/FRIDA-Decisions")   # repo id or a local folder
device = "cuda" if torch.cuda.is_available() else "cpu"
if device == "cpu":
    torch.set_num_threads(min(4, os.cpu_count() or 1))
judge = Judge.from_pretrained(MODEL, device=device)
gpu = f" ({torch.cuda.get_device_name(0)})" if device == "cuda" else ""
print(f"frida-decisions {frida_decisions.__version__}, loaded on {device}{gpu}, {judge.dtype}")


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 6 files:   0%|          | 0/6 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/219 [00:00<?, ?it/s]

frida-decisions 0.4.0, loaded on cpu, torch.float32


In [3]:
SAME = [
    "Точно разные товары.",
    "Скорее разные товары.",
    "Неясно, тот же товар или другой.",
    "Скорее один и тот же товар.",
    "Точно один и тот же товар.",
]
# score is 0..4. On prose cards a real match landed near 3.2 and a different
# model near 0.2, so 2.5 / 1.5 leave a band for "look at this pair".
SAME_MIN = 2.5
DIFFERENT_MAX = 1.5


def request_for(text: str) -> dict:
    return {
        "state": text,
        "questions": {
            "same": {"type": "score", "instructions": "Это один и тот же товар в двух карточках?",
                     "criteria": SAME},
            "brand": {"type": "noul", "instructions": "Бренд в двух карточках один и тот же?"},
            "model": {"type": "noul", "instructions": "Артикул или модель в двух карточках совпадают?"},
        },
    }


def decide(answers: dict) -> dict:
    """The score makes the call. The yes/no answers are reported, not vetoes:
    a field question can fire on a paraphrase even when the number matches.
    """
    score = answers["same"]["score"]
    if score >= SAME_MIN:
        outcome = "same"
    elif score < DIFFERENT_MAX:
        outcome = "different"
    else:
        outcome = "review"
    return {
        "outcome": outcome,
        "score": score,
        "brand": answers["brand"]["noul"],
        "model": answers["model"]["noul"],
    }


## The score makes the call

`same` at or above 2.5 is a match, below 1.5 is not, and the band between them waits for a person. The yes/no questions are printed beside the score. They are not a veto: a paraphrase can move them without moving the score.


In [4]:
def run(judge, rows: list[dict]) -> None:
    hits = decided = 0
    for row in rows:
        response = judge(request_for(row["text"]))
        decision = decide(response["answers"])
        match = decision["outcome"] == row["gold"]
        hits += match
        decided += decision["outcome"] != "review"
        print(f"{'ok' if match else 'miss':4} got {decision['outcome']:10} gold {row['gold']:10} "
              f"score {decision['score']:.2f}  brand {decision['brand']:.2f}  model {decision['model']:.2f}")
        print(f"     {row['text'].replace(chr(10), ' | ')}")
    print(f"\n{hits}/{len(rows)} outcomes match the label, {decided}/{len(rows)} decided without review")


In [5]:
def load_rows():
    import urllib.request
    path = Path("data.json")
    if not path.exists():
        url = ("https://raw.githubusercontent.com/ai-forever/FRIDA-Decisions/"
               "grok/examples-entity-alignment/examples/entity_alignment/data.json")
        path.write_bytes(urllib.request.urlopen(url, timeout=60).read())
    return json.loads(path.read_text(encoding="utf-8"))

run(judge, load_rows())


ok   got same       gold same       score 3.39  brand 0.98  model 0.99
     Карточка А: электрочайник Philips, артикул HD9350, объём 1.7 литра. | Карточка Б: чайник Philips модели HD9350 на 1.7 литра.
ok   got same       gold same       score 3.22  brand 0.95  model 0.96
     Карточка А: наушники Sony, артикул WH-1000XM5, цвет чёрный. | Карточка Б: беспроводные наушники Sony WH-1000XM5, чёрные.
ok   got different  gold different  score 0.93  brand 0.23  model 0.18
     Карточка А: электрочайник Philips, артикул HD9350, объём 1.7 литра. | Карточка Б: электрочайник Philips, артикул HD4649, объём 1.5 литра.
ok   got different  gold different  score 0.90  brand 0.01  model 0.00
     Карточка А: молоко «Домик в деревне» 3.2%, 1 литр. | Карточка Б: стол IKEA Linnmon, белый, ширина 100 см.
ok   got different  gold different  score 1.01  brand 0.19  model 0.10
     Карточка А: стул IKEA Adde, белый. | Карточка Б: стол IKEA Linnmon, белый, ширина 100 см.
ok   got different  gold different  scor